In [1]:
import pandas as pd
import numpy as np
from pyproj import Transformer
from google.colab import files

In [2]:
df_accidentes = pd.read_csv('/content/Accidentes_trafico_madrid_2025.csv', on_bad_lines='skip', sep=';')
df_accidentes.head(5)

,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorológico,tipo_vehiculo,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga
0,2025S000056,01/01/2025,0:49:00,CALL. LOPEZ DE HOYOS / CALL. ROS DE OLANO,140,5,CHAMARTÍN,Colisión fronto-lateral,Despejado,Ciclomotor,Conductor,De 30 a 34 años,Hombre,7.0,Asistencia sanitaria sólo en el lugar del acci...,442966.0,4477385.0,N,NaN
1,2025S000056,01/01/2025,0:49:00,CALL. LOPEZ DE HOYOS / CALL. ROS DE OLANO,140,5,CHAMARTÍN,Colisión fronto-lateral,Despejado,Turismo,Conductor,De 40 a 44 años,Hombre,14.0,Sin asistencia sanitaria,442966.0,4477385.0,N,NaN
2,2025S000056,01/01/2025,0:49:00,CALL. LOPEZ DE HOYOS / CALL. ROS DE OLANO,140,5,CHAMARTÍN,Colisión fronto-lateral,Despejado,Turismo,Pasajero,De 40 a 44 años,Mujer,14.0,Sin asistencia sanitaria,442966.0,4477385.0,N,NaN
3,2025S000057,01/01/2025,2:20:00,GTA. RIO ZANCARA / CALL. ARROYO DE POZUELO,1,9,MONCLOA-ARAVACA,Colisión lateral,Despejado,Turismo,Conductor,De 21 a 24 años,Hombre,NaN,NaN,433911.0,4478290.0,N,NaN
4,2025S000057,01/01/2025,2:20:00,GTA. RIO ZANCARA / CALL. ARROYO DE POZUELO,1,9,MONCLOA-ARAVACA,Colisión lateral,Despejado,Turismo,Conductor,De 45 a 49 años,Hombre,NaN,NaN,433911.0,4478290.0,N,NaN


In [3]:
#Ver información general del dataset
df_accidentes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51067 entries, 0 to 51066
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   num_expediente        51067 non-null  object 
 1   fecha                 51067 non-null  object 
 2   hora                  51067 non-null  object 
 3   localizacion          51064 non-null  object 
 4   numero                51067 non-null  object 
 5   cod_distrito          51067 non-null  int64  
 6   distrito              51067 non-null  object 
 7   tipo_accidente        51067 non-null  object 
 8   estado_meteorológico  45017 non-null  object 
 9   tipo_vehiculo         50638 non-null  object 
 10  tipo_persona          51067 non-null  object 
 11  rango_edad            51067 non-null  object 
 12  sexo                  51067 non-null  object 
 13  cod_lesividad         28442 non-null  float64
 14  lesividad             28442 non-null  object 
 15  coordenada_x_utm   

In [4]:
#Eliminar los valores duplicados y resetar el indice
df_accidentes = df_accidentes.drop_duplicates().reset_index(drop=True)

In [5]:
#Realizar la conversión de las columnas en formato UTM a longitud y latitud
transformer = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)
mask = df_accidentes['coordenada_x_utm'].notna() & df_accidentes['coordenada_y_utm'].notna()
lon, lat = transformer.transform(
    df_accidentes.loc[mask, 'coordenada_x_utm'].values,
    df_accidentes.loc[mask, 'coordenada_y_utm'].values
)

df_accidentes.loc[mask, 'longitud'] = lon
df_accidentes.loc[mask, 'latitud'] = lat

df_accidentes[['coordenada_x_utm', 'coordenada_y_utm', 'longitud', 'latitud']]

,coordenada_x_utm,coordenada_y_utm,longitud,latitud
0,442966.0,4477385.0,-3.672546,40.445163
1,442966.0,4477385.0,-3.672546,40.445163
2,442966.0,4477385.0,-3.672546,40.445163
3,433911.0,4478290.0,-3.779408,40.452645
4,433911.0,4478290.0,-3.779408,40.452645
...,...,...,...,...
48997,441209.0,4473174.0,-3.692874,40.407106
48998,437123.0,4472942.0,-3.741002,40.404717
48999,437123.0,4472942.0,-3.741002,40.404717
49000,445776.0,4474934.0,-3.639203,40.423271


In [6]:
#Ver valores nulos de las nueva columnas
df_accidentes[['longitud','latitud']].isnull().sum()

,0
longitud,14
latitud,14


**Conclusión:**

No se eliminan estas 14 filas porque representan un porcentaje muy pequeño del dataset. Mantenerlas evita perder información útil de otras columnas.

In [7]:
#Descargar el csv tras haber hecho las operaciones necesarias
nombre_archivo = 'Accidentes_trafico_Madrid_2025_con_coordenadas.csv'
df_accidentes.to_csv(nombre_archivo, index=False, sep=';', decimal=',', encoding='utf-8-sig')
files.download(nombre_archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>